# Ejecutar con kernel NPU4 o superior

# Resnet 
## Version Apple Mx
#### Estos cambios permiten usar la GPU MPS de Apple

Se configura VSCode para que use automaticamente, si es posible, GPUs
Configuracion : @id:editor.experimentalGpuAcceleration @id:terminal.integrated.gpuAcceleration -> on

Se reemplaza esta linea:

-----------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

-----------------------------------------------

por esto:

-----------------------------------------------

### Configuración optimizada para Mac M4
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Usando aceleración MPS (GPU M4)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ Usando CUDA")
else:
    device = torch.device("cpu")
    print("⚠️ Usando CPU solamente")

print(f"Dispositivo seleccionado: {device}")

-----------------------------------------------

Finalmente, se cambia:

-----------------------------------------------

epoch_acc = epoch_phase_running_corrects.double() / len(datasets[phase])

-----------------------------------------------

por esto otro:

-----------------------------------------------

### Conversión compatible con MPS

if device.type == 'mps':
    epoch_acc = epoch_phase_running_corrects.float() / len(datasets[phase])
else:
    epoch_acc = epoch_phase_running_corrects.double() / len(datasets[phase])

-----------------------------------------------

In [1]:
# Se agrega este codigo para evitar warnings de optuna
import time
import warnings
warnings.filterwarnings("ignore")

# Configuración adicional del experimento
proyecto = 'petfinder' # No cambiar nombre porque se usa para la base de optun
experimento = 'original' # Este nombre se cambia para describir el experimento

# epochs -> ciclos
ciclos = 2


In [2]:
# ===== CONFIGURACIÓN DE LOGGING PROFESIONAL =====
import logging
import sys
from datetime import datetime
import os

def setup_professional_logging(proyecto, experimento):
    """
    Configura un sistema de logging profesional para el experimento de ML
    
    Args:
        proyecto (str): Nombre del proyecto
        experimento (str): Nombre del experimento específico
    
    Returns:
        logging.Logger: Logger configurado
    """
    
    # Crear directorio de logs si no existe
    log_dir = f"../logs/{proyecto}"
    os.makedirs(log_dir, exist_ok=True)
    
    # Crear nombre único para el archivo de log
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_filename = f"{log_dir}/{experimento}_{timestamp}.log"
    
    # Configurar formato detallado
    detailed_formatter = logging.Formatter(
        '%(asctime)s | %(name)s | %(levelname)8s | %(funcName)s:%(lineno)d | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Configurar formato para consola (más compacto)
    console_formatter = logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s',
        datefmt='%H:%M:%S'
    )
    
    # Crear logger principal
    logger_name = f"{proyecto}_{experimento}"
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    
    # Limpiar handlers previos si existen
    logger.handlers.clear()
    
    # Handler para archivo (nivel DEBUG - todo)
    file_handler = logging.FileHandler(log_filename, mode='w', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(detailed_formatter)
    logger.addHandler(file_handler)
    
    # Handler para consola (nivel INFO - solo información importante)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)
    
    # Evitar propagación a logger raíz
    logger.propagate = False
    
    # Log inicial del sistema
    logger.info("="*80)
    logger.info(f"🚀 INICIANDO EXPERIMENTO: {proyecto} - {experimento}")
    logger.info(f"📁 Log file: {log_filename}")
    logger.info(f"🕐 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.info("="*80)
    
    return logger

# Configurar logging para este experimento
logger = setup_professional_logging(proyecto, experimento)

# Log de configuración inicial
logger.info(f"📊 Configuración del experimento:")
logger.info(f"   ├── Proyecto: {proyecto}")
logger.info(f"   ├── Experimento: {experimento}")
logger.info(f"   └── Ciclos de entrenamiento: {ciclos}")

# Log del entorno de ejecución
logger.debug(f"🐍 Python version: {sys.version}")
logger.debug(f"💻 Working directory: {os.getcwd()}")
logger.debug(f"⚠️  Warnings filtrados: Activo")

01:45:47 | INFO | ================================================================================
01:45:47 | INFO | 🚀 INICIANDO EXPERIMENTO: petfinder - original
01:45:47 | INFO | 📁 Log file: ../logs/petfinder/original_20250817_014547.log
01:45:47 | INFO | 🕐 Timestamp: 2025-08-17 01:45:47
01:45:47 | INFO | ================================================================================
01:45:47 | INFO | 🚀 INICIANDO EXPERIMENTO: petfinder - original
01:45:47 | INFO | 📁 Log file: ../logs/petfinder/original_20250817_014547.log
01:45:47 | INFO | 🕐 Timestamp: 2025-08-17 01:45:47
01:45:47 | INFO | ================================================================================
01:45:47 | INFO | 📊 Configuración del experimento:
01:45:47 | INFO |    ├── Proyecto: petfinder
01:45:47 | INFO |    ├── Experimento: original
01:45:47 | INFO | 📊 Configuración del experimento:
01:45:47 | INFO |    ├── Proyecto: petfinder
01:45:47 | INFO |    ├── Experimento: original
01:45:47 | INFO |    └── Ciclos d

### **FUENTES**:

PetFinder Kaggle:

https://www.kaggle.com/competitions/petfinder-adoption-prediction/data

First Tutorial:

https://towardsdatascience.com/how-to-train-an-image-classifier-in-pytorch-and-use-it-to-perform-basic-inference-on-single-images-99465a1e9bf5

---> no esta disponible <---


Second Deep Tutorial:

https://rumn.medium.com/part-1-ultimate-guide-to-fine-tuning-in-pytorch-pre-trained-model-and-its-configuration-8990194b71e

Logo Recognition API:

https://heartbeat.comet.ml/logo-recognition-ios-application-using-machine-learning-and-flask-api-aec4eff3be11

Hybrid (multimodal) neural network architecture : Combination of tabular, textual and image inputs:

https://medium.com/@dave.cote.msc/hybrid-multimodal-neural-network-architecture-combination-of-tabular-textual-and-image-inputs-7460a4f82a2e



### **INDICACIONES PREVIAS**:

+ **Git**:
    + Clonamos el repo: root de todos los repos y ponemos git clone "url_repo"
    + Hacemos el checkout de la rama main: git checkout -b new-branch

+ **Poetry**:
    + Instalamos poetry: https://python-poetry.org/docs/
    + Realizamos un Update del pyproject: poetry update
    + Activamos el entorno que creo poetry: poetry shell --> no soportado
    + Intentamos correr una celda, si nos pide seleccionar el environment y no lo vemos en la lista, cerrar y volver abrir VSC

+ **Torch y CUDA**:
    + Verificar que versión pide torch:
        + Versión de torch instalada: poetry show (en mi caso la 1.13.1)
        + Buscar la versión correspondiente en la documentación: https://pytorch.org/get-started/previous-versions/  (en mi caso el 11.7)
    + Instalar CUDA para Torch (buscar la versión correspondiente de CUDA): https://developer.nvidia.com/cuda-11-7-0-download-archive
    + Verificar que CUDA esté funcional: correr en una celda torch.cuda.is_available()

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import os
import shutil
import time
import copy
import datetime
from tqdm import tqdm

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.autograd import Variable
import torch.nn.functional as F

from joblib import load, dump

# MLflow para tracking de experimentos
import mlflow
import mlflow.pytorch
import mlflow.optuna

from utils import plot_confusion_matrix

# ===== CONFIGURACIÓN DE MLFLOW =====
# Configurar MLflow tracking
mlflow_tracking_uri = f"file://{os.path.abspath('../work')}/mlruns"
mlflow.set_tracking_uri(mlflow_tracking_uri)

# Configurar experimento MLflow
experiment_name = f"{proyecto}_{experimento}"
try:
    experiment = mlflow.create_experiment(experiment_name)
    logger.info(f"🆕 Experimento MLflow creado: {experiment_name}")
except mlflow.exceptions.MlflowException:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    logger.info(f"📂 Experimento MLflow existente: {experiment_name}")

mlflow.set_experiment(experiment_name)
logger.info(f"🔬 MLFLOW CONFIGURADO")
logger.info(f"   ├── Tracking URI: {mlflow_tracking_uri}")
logger.info(f"   ├── Experimento: {experiment_name}")
logger.info(f"   └── Experiment ID: {experiment.experiment_id if hasattr(experiment, 'experiment_id') else 'N/A'}")

# Verificamos que CUDA está funcional
print(f'Disponibilidad CUDA: {torch.cuda.is_available()}')
print(f'Disponibilidad MPS: {torch.backends.mps.is_available()}')

# Logging detallado de las capacidades del sistema
logger.info("🔍 ANÁLISIS DEL SISTEMA DE CÓMPUTO")
logger.info(f"   ├── PyTorch version: {torch.__version__}")
logger.info(f"   ├── CUDA disponible: {torch.cuda.is_available()}")
logger.info(f"   ├── MPS disponible: {torch.backends.mps.is_available()}")
logger.info(f"   └── Núcleos CPU: {os.cpu_count()}")

if torch.cuda.is_available():
    logger.debug(f"   ├── CUDA version: {torch.version.cuda}")
    logger.debug(f"   ├── GPUs disponibles: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        logger.debug(f"   ├── GPU {i}: {torch.cuda.get_device_name(i)}")
        logger.debug(f"   └── Memoria GPU {i}: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f}GB")

01:45:50 | INFO | 📂 Experimento MLflow existente: petfinder_original
01:45:50 | INFO | 🔬 MLFLOW CONFIGURADO
01:45:50 | INFO |    ├── Tracking URI: file:///Users/alejandro/Documents/Austral/GitHub/UA_MDM_Labo2/work/mlruns
01:45:50 | INFO |    ├── Experimento: petfinder_original
01:45:50 | INFO | 🔬 MLFLOW CONFIGURADO
01:45:50 | INFO |    ├── Tracking URI: file:///Users/alejandro/Documents/Austral/GitHub/UA_MDM_Labo2/work/mlruns
01:45:50 | INFO |    ├── Experimento: petfinder_original
01:45:50 | INFO |    └── Experiment ID: 162465273007525882
Disponibilidad CUDA: False
Disponibilidad MPS: True
01:45:50 | INFO | 🔍 ANÁLISIS DEL SISTEMA DE CÓMPUTO
01:45:50 | INFO |    ├── PyTorch version: 2.8.0
01:45:50 | INFO |    ├── CUDA disponible: False
01:45:50 | INFO |    ├── MPS disponible: True
01:45:50 | INFO |    └── Núcleos CPU: 10
01:45:50 | INFO |    └── Experiment ID: 162465273007525882
Disponibilidad CUDA: False
Disponibilidad MPS: True
01:45:50 | INFO | 🔍 ANÁLISIS DEL SISTEMA DE CÓMPUTO
01:4

**Seteo el Modelo**

Teoría de Resnet: https://towardsdatascience.com/introduction-to-resnets-c0a830a288a4

In [4]:
# Importo modelo ResNet entrenado en Imagenet
logger.info("🧠 CONFIGURACIÓN DEL MODELO")
logger.info("   ├── Cargando ResNet50 preentrenado...")

resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
logger.info("   ├── ✅ ResNet50 cargado exitosamente")

# Modificar la última capa para adaptarse a tu problema específico
num_ftrs = resnet50.fc.in_features
resnet50.fc = torch.nn.Linear(num_ftrs, 5) # Clasificación 5 clases
logger.info(f"   ├── Capa final modificada: {num_ftrs} → 5 clases")

# Configuro para usar cuda si está disponible

# Configuración optimizada para Mac Mx
logger.info("   ├── Detectando dispositivo óptimo...")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Usando aceleración MPS (GPU Mx Apple)")
    logger.info("   ├── 🚀 Dispositivo seleccionado: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ Usando CUDA")
    logger.info(f"   ├── 🚀 Dispositivo seleccionado: CUDA (GPU {torch.cuda.get_device_name()})")
else:
    device = torch.device("cpu")
    print("⚠️ Usando CPU solamente")
    logger.warning("   ├── ⚠️  Dispositivo seleccionado: CPU (sin aceleración)")

print(f"Dispositivo seleccionado: {device}")
logger.info(f"   └── Dispositivo final: {device}")

logger.info("   ├── Transfiriendo modelo al dispositivo...")
resnet50 = resnet50.to(device)
logger.info("   ├── ✅ Modelo transferido exitosamente")

# Instancio del criterio de pérdida CrossEntropyLoss
criterion = nn.CrossEntropyLoss()
logger.info("   └── ✅ Criterio de pérdida configurado: CrossEntropyLoss")



01:45:50 | INFO | 🧠 CONFIGURACIÓN DEL MODELO
01:45:50 | INFO |    ├── Cargando ResNet50 preentrenado...
01:45:50 | INFO |    ├── Cargando ResNet50 preentrenado...
01:45:50 | INFO |    ├── ✅ ResNet50 cargado exitosamente
01:45:50 | INFO |    ├── Capa final modificada: 2048 → 5 clases
01:45:50 | INFO |    ├── Detectando dispositivo óptimo...
✅ Usando aceleración MPS (GPU Mx Apple)
01:45:50 | INFO |    ├── 🚀 Dispositivo seleccionado: MPS (Apple Silicon GPU)
Dispositivo seleccionado: mps
01:45:50 | INFO |    └── Dispositivo final: mps
01:45:50 | INFO |    ├── Transfiriendo modelo al dispositivo...
01:45:51 | INFO |    ├── ✅ Modelo transferido exitosamente
01:45:51 | INFO |    └── ✅ Criterio de pérdida configurado: CrossEntropyLoss
01:45:50 | INFO |    ├── ✅ ResNet50 cargado exitosamente
01:45:50 | INFO |    ├── Capa final modificada: 2048 → 5 clases
01:45:50 | INFO |    ├── Detectando dispositivo óptimo...
✅ Usando aceleración MPS (GPU Mx Apple)
01:45:50 | INFO |    ├── 🚀 Dispositivo selec

**Seteo parámetros, directorios y funciones**

In [5]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train_images")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, f"work/{proyecto}/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, f"work/{proyecto}/optuna_artifacts")

MODEL_NAME = '05 ResNet'
MODEL_VERSION = '1.0.0'

# Parametros y variables
CREATE_PYTORCH_DIRECTORIES = 1
SEED = 42
BATCH_SIZE = 50
TEST_SIZE = 0.2
IMAGE_SIZE = 299
CPU_CORES = os.cpu_count()

# Logging de configuración de parámetros
logger.info("⚙️  CONFIGURACIÓN DE PARÁMETROS")
logger.info(f"   ├── Modelo: {MODEL_NAME} v{MODEL_VERSION}")
logger.info(f"   ├── Batch size: {BATCH_SIZE}")
logger.info(f"   ├── Test size: {TEST_SIZE} ({TEST_SIZE*100}%)")
logger.info(f"   ├── Tamaño de imagen: {IMAGE_SIZE}x{IMAGE_SIZE}")
logger.info(f"   ├── Semilla aleatoria: {SEED}")
logger.info(f"   ├── CPU cores disponibles: {CPU_CORES}")
logger.info(f"   └── Crear directorios PyTorch: {'Sí' if CREATE_PYTORCH_DIRECTORIES else 'No'}")

logger.info("📁 CONFIGURACIÓN DE RUTAS")
logger.info(f"   ├── BASE_DIR: {BASE_DIR}")
logger.info(f"   ├── CSV de entrenamiento: {PATH_TO_TRAIN}")
logger.info(f"   ├── Directorio de imágenes: {PATH_TO_IMAGES_DIR}")
logger.info(f"   ├── Archivos temporales: {PATH_TO_TEMP_FILES}")
logger.info(f"   └── Artefactos Optuna: {PATH_TO_OPTUNA_ARTIFACTS}")

# Verificar que archivos/directorios existen
logger.debug("🔍 Verificando existencia de archivos críticos:")
if os.path.exists(PATH_TO_TRAIN):
    logger.debug(f"   ├── ✅ CSV encontrado: {PATH_TO_TRAIN}")
else:
    logger.error(f"   ├── ❌ CSV NO encontrado: {PATH_TO_TRAIN}")
    
if os.path.exists(PATH_TO_IMAGES_DIR):
    logger.debug(f"   └── ✅ Directorio de imágenes encontrado: {PATH_TO_IMAGES_DIR}")
else:
    logger.error(f"   └── ❌ Directorio de imágenes NO encontrado: {PATH_TO_IMAGES_DIR}")

# Armo el nuevo directorio de train
new_train_directory = os.path.join(BASE_DIR, 'work/train_images_classes')
os.makedirs(new_train_directory, exist_ok=True) # si ya existe el nombre, lo deja como está

# Armo el nuevo directorio de validación
new_val_directory = os.path.join(BASE_DIR, 'work/val_images_classes')
os.makedirs(new_val_directory, exist_ok=True)

# Crear directorios necesarios para artefactos
os.makedirs(PATH_TO_TEMP_FILES, exist_ok=True)
os.makedirs(PATH_TO_OPTUNA_ARTIFACTS, exist_ok=True)

logger.info("📂 DIRECTORIOS DE TRABAJO CREADOS")
logger.info(f"   ├── Entrenamiento: {new_train_directory}")
logger.info(f"   ├── Validación: {new_val_directory}")
logger.info(f"   ├── Archivos temporales: {PATH_TO_TEMP_FILES}")
logger.info(f"   └── Artefactos Optuna: {PATH_TO_OPTUNA_ARTIFACTS}")

# Definir las clases ordenadas
class_names = ['0', '1', '2', '3', '4']
logger.info(f"🏷️  CLASES DEFINIDAS: {class_names}")

# Mapear las etiquetas de las clases a números enteros consecutivos
class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

# Creo las carpetas de clases dentro de los directorios
for clase in class_names: # Una para cada clase
   os.makedirs(os.path.join(new_train_directory, str(clase)), exist_ok=True)
   os.makedirs(os.path.join(new_val_directory, str(clase)), exist_ok=True)



01:45:51 | INFO | ⚙️  CONFIGURACIÓN DE PARÁMETROS
01:45:51 | INFO |    ├── Modelo: 05 ResNet v1.0.0
01:45:51 | INFO |    ├── Batch size: 50
01:45:51 | INFO |    ├── Test size: 0.2 (20.0%)
01:45:51 | INFO |    ├── Tamaño de imagen: 299x299
01:45:51 | INFO |    ├── Semilla aleatoria: 42
01:45:51 | INFO |    ├── CPU cores disponibles: 10
01:45:51 | INFO |    └── Crear directorios PyTorch: Sí
01:45:51 | INFO | 📁 CONFIGURACIÓN DE RUTAS
01:45:51 | INFO |    ├── BASE_DIR: ../
01:45:51 | INFO |    ├── CSV de entrenamiento: ../input/petfinder-adoption-prediction/train/train.csv
01:45:51 | INFO |    ├── Directorio de imágenes: ../input/petfinder-adoption-prediction/train_images
01:45:51 | INFO |    ├── Archivos temporales: ../work/petfinder/optuna_temp_artifacts
01:45:51 | INFO |    ├── Modelo: 05 ResNet v1.0.0
01:45:51 | INFO |    ├── Batch size: 50
01:45:51 | INFO |    ├── Test size: 0.2 (20.0%)
01:45:51 | INFO |    ├── Tamaño de imagen: 299x299
01:45:51 | INFO |    ├── Semilla aleatoria: 42
0


# Funciones para la carga y el preproceso
def resize_to_square(im):
    old_size = im.shape[:2] # old_size is in (height, width) format
    # Calcula el factor de escala necesario para redimensionar la imagen de manera que el lado más largo tenga el tamaño deseado 
    ratio = float(IMAGE_SIZE)/max(old_size)
    # Calcula las nuevas dimensiones de la imagen 
    new_size = tuple([int(x*ratio) for x in old_size])
    # Redimensiona la imagen con el nuevo tamaño
    im = cv2.resize(im, (new_size[1], new_size[0]))
    # Calcula las diferencias de tamaño y agrega pixeles (color negro) en los extremos para que quede centrada y cuadrada 
    delta_w = IMAGE_SIZE - new_size[1]
    delta_h = IMAGE_SIZE - new_size[0]
    top, bottom = delta_h//2, delta_h-(delta_h//2)
    left, right = delta_w//2, delta_w-(delta_w//2)
    color = [0, 0, 0]
    new_image = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT,value=color)
    return new_image


def load_image(pet_id):
    path_to_image = os.path.join(PATH_TO_IMAGES_DIR, f'{pet_id}-1.jpg') # Irá a la primera imagen de la mascota
    image = cv2.imread(path_to_image)
    # Convierte la imagen de BGR a RGB porque estos modelos esperan ese orden de canales
    image = cv2.convertScaleAbs(image)
    image= cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    new_image = resize_to_square(image)
    return new_image


In [6]:

def visualize_pet(pet_id):
    path_to_image = os.path.join(PATH_TO_IMAGES_DIR, f'{pet_id}-1.jpg') # Irá a la primera imagen de la mascota
    # Cargar la imagen
    image_to_show = cv2.imread(path_to_image)
    # Convertir a formato RGB
    image_to_show = cv2.cvtColor(image_to_show, cv2.COLOR_BGR2RGB)
    # Visualizar la imagen
    plt.imshow(image_to_show)
    plt.axis('off')  # No mostrar los ejes
    plt.show()

def visualize_image(image):
    # Convierte la imagen a un formato de enteros (CV_8U)
    image = cv2.convertScaleAbs(image)
    image= cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # Visualizar la imagen
    plt.imshow(image.astype(np.uint8))
    plt.axis('off')  # No mostrar los ejes
    plt.show()


**Cargo y Proceso Data**

Nota: Pytorch necesita que estén las imágenes en los distintos directorios según su clase y su participación en el training

In [7]:
# Cargo
train_df = pd.read_csv(PATH_TO_TRAIN)

# Split para validación
train_data, val_data = train_test_split(train_df,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = train_df.AdoptionSpeed)

# if CREATE_PYTORCH_DIRECTORIES == 1:  # Ya fue ejecutado al menos una vez, por lo que las carpetas existen
if CREATE_PYTORCH_DIRECTORIES == 0: # Poner en 0 si ya tengo las carpetas train_images_classes y val_images_classes con las imágenes copiadas
    # Función para copiar las imágenes a los directorios correspondientes
    def copy_imag(data, directorio_destino):
        for index, row in data.iterrows():
            petID = row['PetID']
            adoption_speed = row['AdoptionSpeed']
            
            # Nombre del archivo de imagen
            nombre_archivo = f"{petID}-1.jpg"
            
            # Ruta completa de la imagen de origen
            ruta_origen = os.path.join(PATH_TO_IMAGES_DIR, nombre_archivo)
            
            # Ruta completa del directorio de destino
            ruta_destino = os.path.join(directorio_destino, str(adoption_speed), nombre_archivo)
            
            # Verificar si el archivo de origen existe
            if os.path.exists(ruta_origen):
                # Copiar el archivo de origen al directorio de destino
                shutil.copy2(ruta_origen, ruta_destino)
        print("Completada la copia a: ",str(directorio_destino))

    # Copiar las imágenes al directorio de train
    copy_imag(train_data, new_train_directory)

    # Copiar las imágenes al directorio de val
    copy_imag(val_data, new_val_directory)

    print("Proceso completado.")

In [8]:
# Genero los DataLoaders
def create_dataloaders(train_directory, val_directory, batch_size, num_workers):
    # Transformaciones de imagen para el conjunto de entrenamiento
    train_transforms = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Transformaciones de imagen para el conjunto de validación (sin data augment)
    val_transforms = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Crear conjuntos de datos para el conjunto de entrenamiento y validación
    conjunto_entrenamiento = datasets.ImageFolder(train_directory, transform=train_transforms)
    conjunto_validacion = datasets.ImageFolder(val_directory, transform=val_transforms)

    # Asignar las clases ordenadas al conjunto de datos
    conjunto_entrenamiento.class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}
    conjunto_validacion.class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

    # Crear dataloaders para el conjunto de entrenamiento y validación
    train_dataloader = torch.utils.data.DataLoader(conjunto_entrenamiento, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_dataloader = torch.utils.data.DataLoader(conjunto_validacion, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_dataloader, val_dataloader

# Aplico las funcion de los DataLoaders
logger.info("🔄 CREANDO DATALOADERS")
logger.info(f"   ├── Directorio entrenamiento: {new_train_directory}")
logger.info(f"   ├── Directorio validación: {new_val_directory}")
logger.info(f"   ├── Batch size: {BATCH_SIZE}")
logger.info(f"   └── Workers: {CPU_CORES}")

train_dataloader, val_dataloader = create_dataloaders(new_train_directory , new_val_directory , BATCH_SIZE, CPU_CORES)

logger.info("   ├── ✅ DataLoaders creados exitosamente")
logger.info(f"   ├── Batches de entrenamiento: {len(train_dataloader)}")
logger.info(f"   ├── Batches de validación: {len(val_dataloader)}")
logger.info(f"   ├── Muestras de entrenamiento: {len(train_dataloader.dataset)}")
logger.info(f"   └── Muestras de validación: {len(val_dataloader.dataset)}")

01:45:51 | INFO | 🔄 CREANDO DATALOADERS
01:45:51 | INFO |    ├── Directorio entrenamiento: ../work/train_images_classes
01:45:51 | INFO |    ├── Directorio validación: ../work/val_images_classes
01:45:51 | INFO |    ├── Batch size: 50
01:45:51 | INFO |    └── Workers: 10
01:45:51 | INFO |    ├── Directorio entrenamiento: ../work/train_images_classes
01:45:51 | INFO |    ├── Directorio validación: ../work/val_images_classes
01:45:51 | INFO |    ├── Batch size: 50
01:45:51 | INFO |    └── Workers: 10
01:45:51 | INFO |    ├── ✅ DataLoaders creados exitosamente
01:45:51 | INFO |    ├── Batches de entrenamiento: 235
01:45:51 | INFO |    ├── Batches de validación: 59
01:45:51 | INFO |    ├── Muestras de entrenamiento: 11721
01:45:51 | INFO |    └── Muestras de validación: 2931
01:45:51 | INFO |    ├── ✅ DataLoaders creados exitosamente
01:45:51 | INFO |    ├── Batches de entrenamiento: 235
01:45:51 | INFO |    ├── Batches de validación: 59
01:45:51 | INFO |    ├── Muestras de entrenamiento: 

In [9]:
#Genero una lista de PetIDs con imagen en el orden en que aparecen en el data loader
test_sample_ids = [i[0].split('/')[-1].split('-')[0] for i in val_dataloader.dataset.samples]

logger.info("🆔 IDs DE MUESTRAS EXTRAÍDOS")
logger.info(f"   ├── Total IDs extraídos: {len(test_sample_ids)}")
logger.info(f"   ├── Primeros 5 IDs: {test_sample_ids[:5]}")
logger.info(f"   └── Últimos 5 IDs: {test_sample_ids[-5:]}")

01:45:51 | INFO | 🆔 IDs DE MUESTRAS EXTRAÍDOS
01:45:51 | INFO |    ├── Total IDs extraídos: 2931
01:45:51 | INFO |    ├── Primeros 5 IDs: ['015da9e87', '022606901', '02f89bdcb', '0cf7fae9d', '0e922caab']
01:45:51 | INFO |    └── Últimos 5 IDs: ['ff2cf88a0', 'ff498c903', 'ff50c6171', 'ff5e30380', 'ffa5c6c35']
01:45:51 | INFO |    ├── Total IDs extraídos: 2931
01:45:51 | INFO |    ├── Primeros 5 IDs: ['015da9e87', '022606901', '02f89bdcb', '0cf7fae9d', '0e922caab']
01:45:51 | INFO |    └── Últimos 5 IDs: ['ff2cf88a0', 'ff498c903', 'ff50c6171', 'ff5e30380', 'ffa5c6c35']


**Entreno**

In [10]:
def train_val(model, criterion, dataloaders, datasets, device, num_epochs=20, lr=0.001, momentum = 0.9 ,trial=None):
    
    # Log del inicio del entrenamiento
    trial_info = f"Trial {trial.number}" if trial else "Entrenamiento único"
    logger.info("="*80)
    logger.info(f"🚀 INICIANDO ENTRENAMIENTO - {trial_info}")
    logger.info(f"   ├── Learning rate: {lr}")
    logger.info(f"   ├── Momentum: {momentum}")
    logger.info(f"   ├── Épocas: {num_epochs}")
    logger.info(f"   ├── Dispositivo: {device}")
    logger.info(f"   ├── Tamaño entrenamiento: {len(datasets['train'])} muestras")
    logger.info(f"   └── Tamaño validación: {len(datasets['val'])} muestras")
    
    # Instancio Stochastic Gradient Descent (SGD): Defino el parámetro del Learning Rate (define "el paso" en que avanzan los pesos en cada iteración) y el Momentum (pone innercia a la dirección del gradiente descendiente para que no cambie de dirección en minimos locales)
    optimizer = optim.SGD(resnet50.parameters(), lr=lr, momentum=momentum) # Parámetros default del SGD
    logger.debug(f"   ├── Optimizador configurado: SGD(lr={lr}, momentum={momentum})")
    
    #Inicializo variables
    since = time.time()
    logger.debug(f"   └── Timer iniciado: {datetime.datetime.fromtimestamp(since).strftime('%H:%M:%S')}")

    #Inicializo variable de mejor kappa entre trials
    try:
        #Intento obtener el mejor kappa de optuna
        previous_best = study.best_value
        logger.debug(f"   ├── Mejor kappa previo: {previous_best:.4f}")
    except:
        #Si no hay, seteo -999
        previous_best = -999
        logger.debug(f"   ├── Sin kappa previo, usando: {previous_best}")

    #Inicializo variables de mejor modelo y mejor accuracy y mejor kappa de este trial
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa =  -999
    logger.debug(f"   └── Variables de mejor modelo inicializadas")

    # ===== MLFLOW: Log parámetros del entrenamiento =====
    if trial is not None:
        # Si es trial de Optuna, crear run child
        run_name = f"trial_{trial.number}"
        tags = {"trial_number": trial.number, "optimizer": "SGD", "model": "ResNet50"}
        use_nested = True
    else:
        # Si es entrenamiento único, usar run existente o crear nested
        run_name = f"single_run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
        tags = {"optimizer": "SGD", "model": "ResNet50", "type": "single_run"}
        use_nested = True  # Siempre usar nested para evitar conflictos
    
    with mlflow.start_run(run_name=run_name, tags=tags, nested=use_nested):
        # Log parámetros
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("momentum", momentum)
        mlflow.log_param("num_epochs", num_epochs)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("device", str(device))
        mlflow.log_param("train_samples", len(datasets['train']))
        mlflow.log_param("val_samples", len(datasets['val']))
        mlflow.log_param("image_size", IMAGE_SIZE)
        
        if trial:
            mlflow.log_param("trial_number", trial.number)
            mlflow.log_param("optuna_study", trial.study.study_name)


        for epoch in range(num_epochs):
            epoch_start_time = time.time()
            
            logger.info(f"")
            logger.info(f"📊 ÉPOCA {epoch+1}/{num_epochs}")
            logger.info(f"   └── Tiempo actual: {datetime.datetime.now().strftime('%H:%M:%S')}")
            
            print('Epoch {}/{}'.format(epoch, num_epochs - 1))
            print('-' * 10)
            
            #Inicializo listas de kappa true y predicted y scores para esta epoch
            epoch_kappa_labels_true = []
            epoch_kappa_labels_predicted = []
            epoch_output_scores = []

            #Cada epoch tiene una fase de entrenamiento y validación
            for phase in ['train', 'val']:
                phase_start_time = time.time()
                
                if phase == 'train':
                    model.train()  # Set model to training mode
                    logger.debug(f"      ├── 🏋️  Fase ENTRENAMIENTO iniciada")
                else:
                    model.eval()   # Set model to evaluate mode
                    logger.debug(f"      ├── 🧪 Fase VALIDACIÓN iniciada")

                #Inicializo variables de loss y accuracy para esta fase de epoch
                epoch_phase_running_loss = 0.0
                epoch_phase_running_corrects = 0
                
                # Variables para tracking detallado
                batch_count = 0
                total_batches = len(dataloaders[phase])
                log_interval = max(1, total_batches // 10)  # Log cada 10% del progreso

                # Itero sobre los datos.
                for inputs, labels in tqdm(dataloaders[phase]):
                    batch_count += 1
                    
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # Zero the parameter gradients
                    optimizer.zero_grad()

                    # Forward
                    # Track history if only in train
                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        # Backward + optimize only if in training phase
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()
                        elif phase == 'val':
                            #Agrego los valores de kappa true y predicted para cada batch en validación
                            epoch_kappa_labels_true.extend(labels.cpu().numpy().tolist())
                            epoch_kappa_labels_predicted.extend(preds.cpu().numpy().tolist())
                            outputs_np = outputs.cpu().numpy()
                            epoch_output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])

                    # Statistics for each phase
                    epoch_phase_running_loss += loss.item() * inputs.size(0)
                    epoch_phase_running_corrects += torch.sum(preds == labels.data)
                    
                    # Log progreso periódico por batch
                    if batch_count % log_interval == 0 or batch_count == total_batches:
                        batch_acc = torch.sum(preds == labels.data).float() / inputs.size(0)
                        progress_pct = (batch_count / total_batches) * 100
                        logger.debug(f"         ├── Batch {batch_count}/{total_batches} ({progress_pct:.1f}%) - Loss: {loss.item():.4f}, Acc: {batch_acc:.4f}")
                    
                    #END OF BATCH
                
                epoch_loss = epoch_phase_running_loss / len(datasets[phase])
                # Conversión compatible con MPS
                if device.type == 'mps':
                    epoch_acc = epoch_phase_running_corrects.float() / len(datasets[phase])
                else:
                    epoch_acc = epoch_phase_running_corrects.double() / len(datasets[phase])

                #Calculo el kappa para cada epoch
                if phase == 'train':
                    #overall_train_losses.append(epoch_loss)
                    current_kappa_score = np.nan
                else:
                    #overall_val_losses.append(epoch_loss)
                    current_kappa_score = cohen_kappa_score(epoch_kappa_labels_true,
                                      epoch_kappa_labels_predicted,
                                      weights = 'quadratic')
                
                # Log detallado de métricas por fase
                phase_duration = time.time() - phase_start_time
                logger.info(f"      ├── 📈 {phase.upper()} completado en {phase_duration:.1f}s")
                logger.info(f"      ├── Loss: {epoch_loss:.6f}")
                logger.info(f"      ├── Accuracy: {epoch_acc*100:.2f}%")
                if not np.isnan(current_kappa_score):
                    logger.info(f"      └── Kappa: {current_kappa_score:.6f}")
                else:
                    logger.info(f"      └── Kappa: N/A (entrenamiento)")
                
                # ===== MLFLOW: Log métricas por época y fase =====
                mlflow.log_metric(f"{phase}_loss", epoch_loss, step=epoch)
                mlflow.log_metric(f"{phase}_accuracy", float(epoch_acc), step=epoch)
                mlflow.log_metric(f"{phase}_duration", phase_duration, step=epoch)
                
                if not np.isnan(current_kappa_score):
                    mlflow.log_metric(f"{phase}_kappa", current_kappa_score, step=epoch)
                        
                print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {current_kappa_score:.3f}')

                # If this is the best Epoch so far -> Deep copy the model
                if phase == 'val' and current_kappa_score > best_kappa:
                    best_acc = epoch_acc
                    best_kappa = current_kappa_score
                    best_model_wts = copy.deepcopy(model.state_dict())
                    
                    logger.info(f"      🏆 ¡NUEVO MEJOR MODELO!")
                    logger.info(f"         ├── Mejor Accuracy: {best_acc*100:.2f}%")
                    logger.info(f"         └── Mejor Kappa: {best_kappa:.6f}")
                    
                    # ===== MLFLOW: Log mejor modelo =====
                    mlflow.log_metric("best_accuracy", float(best_acc))
                    mlflow.log_metric("best_kappa", best_kappa)

                    #Best Epoch within a trial and better than previous trials
                    if trial is not None and best_kappa > previous_best:
                        logger.info(f"         🎯 Mejor que trials anteriores! (prev: {previous_best:.6f})")

                        #Save test dataset with predictions
                        predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
                        predicted_df = pd.DataFrame({'PetID':test_sample_ids,
                                    'pred':epoch_output_scores}).merge(val_data, on='PetID')
                        dump(predicted_df, predicted_filename)
                        logger.debug(f"         ├── Predicciones guardadas: {predicted_filename}")

                        #Generate and save CM 
                        cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
                        plot_confusion_matrix(epoch_kappa_labels_true,epoch_kappa_labels_predicted).write_image(cm_filename)
                        logger.debug(f"         └── Matriz confusión guardada: {cm_filename}")
                        
                        # ===== MLFLOW: Log artefactos del mejor modelo =====
                        try:
                            mlflow.log_artifact(predicted_filename, "predictions")
                            mlflow.log_artifact(cm_filename, "confusion_matrices")
                            logger.debug(f"         ├── MLflow: Artefactos registrados")
                        except Exception as e:
                            logger.warning(f"         ├── MLflow warning: {str(e)}")

                #END OF PHASE

            # Log resumen de la época
            epoch_duration = time.time() - epoch_start_time
            logger.info(f"   ⏱️  Época {epoch+1} completada en {epoch_duration:.1f}s")
            
            #END OF EPOCH

        time_elapsed = time.time() - since
        
        # Log resumen final del entrenamiento
        logger.info("")
        logger.info("🏁 ENTRENAMIENTO COMPLETADO")
        logger.info(f"   ├── ⏱️  Tiempo total: {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s")
        logger.info(f"   ├── 🎯 Mejor Accuracy: {best_acc*100:.2f}%")
        logger.info(f"   ├── 🏆 Mejor Kappa: {best_kappa:.6f}")
        
        if trial:
            logger.info(f"   └── Trial {trial.number} finalizado")
        else:
            logger.info(f"   └── Entrenamiento único finalizado")
        
        print('Training complete in {:.0f}m {:.0f}s'.format(
            time_elapsed // 60, time_elapsed % 60))
        print('Best val Acc: {:.2f}%'.format(best_acc * 100))

        # Load best model weights
        model.load_state_dict(best_model_wts)
        logger.debug("   └── Mejores pesos del modelo cargados")

        # Save in optuna trial the best test dataset, cm and model weights
        if trial is not None and best_kappa > previous_best:
            logger.info("💾 GUARDANDO ARTEFACTOS DE MEJOR TRIAL")
            
            upload_artifact(trial, predicted_filename, artifact_store)   
            logger.debug(f"   ├── Predicciones subidas: {predicted_filename}")

            upload_artifact(trial, cm_filename, artifact_store)
            logger.debug(f"   ├── Matriz confusión subida: {cm_filename}")

            file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth'
            model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
            torch.save(model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
            upload_artifact(trial, model_path, artifact_store)
            logger.info(f"   └── Modelo guardado y subido: {model_path}")
            
            # ===== MLFLOW: Log modelo PyTorch =====
            try:
                # Crear ejemplo de entrada para la signatura del modelo
                dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
                
                mlflow.pytorch.log_model(
                    model, 
                    "model",
                    input_example=dummy_input.cpu().numpy()
                )
                mlflow.log_artifact(model_path, "saved_models")
                logger.debug(f"   └── MLflow: Modelo PyTorch registrado con signatura")
            except Exception as e:
                logger.warning(f"   └── MLflow warning: {str(e)}")

            # ===== MLFLOW: Log métricas finales y metadatos =====
            mlflow.log_metric("total_training_time", time_elapsed)
            mlflow.log_metric("final_best_accuracy", float(best_acc))
            mlflow.log_metric("final_best_kappa", best_kappa)
            
            # Log tags adicionales
            mlflow.set_tag("training_completed", "true")
            mlflow.set_tag("device_used", str(device))
            
            if trial:
                mlflow.set_tag("is_optuna_trial", "true")
                mlflow.set_tag("trial_number", trial.number)
            else:
                mlflow.set_tag("is_optuna_trial", "false")

    return model,best_kappa

# ===== ENTRENAMIENTO ÚNICO (SIN OPTUNA) =====
with mlflow.start_run(run_name="single_training_run") as single_run:
    # Log configuración del entrenamiento único
    mlflow.log_param("training_type", "single_run")
    mlflow.log_param("num_epochs", ciclos)
    mlflow.set_tag("is_single_training", "true")
    
    best_model,_ = train_val(resnet50, criterion, 
                           dataloaders={'train': train_dataloader, 
                                        'val': val_dataloader}, 
                           datasets={'train': train_data, 'val': val_data}, 
                           device=device, 
                           num_epochs=ciclos)

# Guardo el modelo
logger.info("💾 GUARDANDO MODELO FINAL")

run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()

logger.info(f"   └── ✅ Modelo guardado: {model_path}")
print(f'Modelo guardado en {model_path}')

# ===== MLFLOW: Log modelo final del entrenamiento único =====
with mlflow.start_run(run_id=single_run.info.run_id):
    try:
        # Crear ejemplo de entrada para la signatura del modelo
        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
        
        mlflow.pytorch.log_model(
            best_model, 
            "final_model",
            input_example=dummy_input.cpu().numpy()
        )
        mlflow.log_artifact(model_path, "saved_models")
        mlflow.set_tag("final_model_saved", "true")
        logger.info(f"   └── ✅ Modelo final registrado en MLflow con signatura")
    except Exception as e:
        logger.warning(f"   └── MLflow warning: {str(e)}")

01:45:51 | INFO | ================================================================================
01:45:51 | INFO | 🚀 INICIANDO ENTRENAMIENTO - Entrenamiento único
01:45:51 | INFO |    ├── Learning rate: 0.001
01:45:51 | INFO |    ├── Momentum: 0.9
01:45:51 | INFO |    ├── Épocas: 2
01:45:51 | INFO |    ├── Dispositivo: mps
01:45:51 | INFO |    ├── Tamaño entrenamiento: 11994 muestras
01:45:51 | INFO |    └── Tamaño validación: 2999 muestras
01:45:51 | INFO | 🚀 INICIANDO ENTRENAMIENTO - Entrenamiento único
01:45:51 | INFO |    ├── Learning rate: 0.001
01:45:51 | INFO |    ├── Momentum: 0.9
01:45:51 | INFO |    ├── Épocas: 2
01:45:51 | INFO |    ├── Dispositivo: mps
01:45:51 | INFO |    ├── Tamaño entrenamiento: 11994 muestras
01:45:51 | INFO |    └── Tamaño validación: 2999 muestras
01:45:51 | INFO | 
01:45:51 | INFO | 📊 ÉPOCA 1/2
01:45:51 | INFO |    └── Tiempo actual: 01:45:51
01:45:51 | INFO | 
01:45:51 | INFO | 📊 ÉPOCA 1/2
01:45:51 | INFO |    └── Tiempo actual: 01:45:51
Epoch 0/1

100%|██████████| 235/235 [12:10<00:00,  3.11s/it]

01:58:02 | INFO |       ├── 📈 TRAIN completado en 730.4s
01:58:02 | INFO |       ├── Loss: 1.419391
01:58:02 | INFO |       ├── Accuracy: 31.27%
01:58:02 | INFO |       └── Kappa: N/A (entrenamiento)
01:58:02 | INFO |       ├── Loss: 1.419391
01:58:02 | INFO |       ├── Accuracy: 31.27%
01:58:02 | INFO |       └── Kappa: N/A (entrenamiento)


Train Loss: 1.4194 Acc: 31.27% Kappa: nan


100%|██████████| 59/59 [01:52<00:00,  1.91s/it]

...

Val Loss: 1.3740 Acc: 33.48% Kappa: 0.262
02:13:13 | INFO |       🏆 ¡NUEVO MEJOR MODELO!
02:13:13 | INFO |          ├── Mejor Accuracy: 33.48%
02:13:13 | INFO |          └── Mejor Kappa: 0.262284
02:13:13 | INFO |    ⏱️  Época 2 completada en 797.8s
02:13:13 | INFO | 
02:13:13 | INFO | 🏁 ENTRENAMIENTO COMPLETADO
02:13:13 | INFO |    ├── ⏱️  Tiempo total: 27m 22s
02:13:13 | INFO |    ├── 🎯 Mejor Accuracy: 33.48%
02:13:13 | INFO |          ├── Mejor Accuracy: 33.48%
02:13:13 | INFO |          └── Mejor Kappa: 0.262284
02:13:13 | INFO |    ⏱️  Época 2 completada en 797.8s
02:13:13 | INFO | 
02:13:13 | INFO | 🏁 ENTRENAMIENTO COMPLETADO
02:13:13 | INFO |    ├── ⏱️  Tiempo total: 27m 22s
02:13:13 | INFO |    ├── 🎯 Mejor Accuracy: 33.48%
02:13:13 | INFO |    ├── 🏆 Mejor Kappa: 0.262284
02:13:13 | INFO |    └── Entrenamiento único finalizado
02:13:13 | INFO |    ├── 🏆 Mejor Kappa: 0.262284
02:13:13 | INFO |    └── Entrenamiento único finalizado
Training complete in 27m 22s
Best val Acc: 33.48%

2025/08/17 02:13:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Modelo guardado en ../work/petfinder/optuna_temp_artifacts/05 ResNet_1.0.0_20250817_021313.pth


2025/08/17 02:13:14 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: RuntimeError("slow_conv2d_forward_mps: input(device='cpu') and weight(device=mps:0')  must be on the same device"). To see the full traceback, set the logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)`.
2025/08/17 02:13:19 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: expected scalar type Double but found Float
2025/08/17 02:13:19 WARNING mlflow.models.mo

02:13:19 | INFO |    └── ✅ Modelo final registrado en MLflow con signatura


In [11]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)


def optuna_train(trial):
    
    logger.info("="*60)
    logger.info(f"🔬 INICIANDO OPTUNA TRIAL #{trial.number}")
    
    epochs = trial.suggest_int('epochs', ciclos, ciclos)
    lr = trial.suggest_float('lr', 0.00001, 0.1, log=True)
    momentum = trial.suggest_float('momentum', 0.0, 0.95)
    
    logger.info(f"   ├── Hiperparámetros sugeridos:")
    logger.info(f"   ├──   Epochs: {epochs}")
    logger.info(f"   ├──   Learning Rate: {lr:.6f}")
    logger.info(f"   └──   Momentum: {momentum:.4f}")

    # ===== MLFLOW: Crear run padre para el trial de Optuna =====
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True) as parent_run:
        # Log parámetros del trial a nivel padre
        mlflow.log_param("optuna_trial_number", trial.number)
        mlflow.log_param("suggested_epochs", epochs)
        mlflow.log_param("suggested_lr", lr)
        mlflow.log_param("suggested_momentum", momentum)
        
        # Tags para identificar
        mlflow.set_tag("optuna_trial", "true")
        mlflow.set_tag("trial_number", trial.number)

        _,best_score = train_val(resnet50, criterion,
                           dataloaders={'train': train_dataloader, 
                                        'val': val_dataloader}, 
                           datasets={'train': train_data, 'val': val_data}, 
                           device=device, 
                           num_epochs=epochs,
                           lr=lr,
                           momentum = momentum,
                           trial=trial)

        # Log resultado final del trial
        mlflow.log_metric("trial_final_kappa", best_score)
        mlflow.set_tag("trial_completed", "true")

    logger.info(f"🏁 TRIAL #{trial.number} COMPLETADO - Kappa final: {best_score:.6f}")
    logger.info("="*60)
    
    return(best_score)

In [13]:
logger.info("🔍 INICIANDO OPTIMIZACIÓN OPTUNA")
logger.info(f"   ├── Proyecto: {proyecto}")
logger.info(f"   ├── Modelo: {MODEL_NAME}_{MODEL_VERSION}")
logger.info(f"   ├── Base de datos: ../work/{proyecto}/db.sqlite3")
logger.info(f"   └── Número de trials: 20")

study = optuna.create_study(direction='maximize',
                            storage=f"sqlite:///../work/{proyecto}/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)

# ===== MLFLOW: Log estudio de Optuna =====
with mlflow.start_run(run_name=f"optuna_study_{MODEL_NAME}_{MODEL_VERSION}") as study_run:
    # Log configuración del estudio
    mlflow.log_param("n_trials", 20)
    mlflow.log_param("direction", "maximize")
    mlflow.log_param("study_name", f'{MODEL_NAME}_{MODEL_VERSION}')
    mlflow.log_param("storage", f"sqlite:///../work/{proyecto}/db.sqlite3")
    
    # Tags para identificar el estudio
    mlflow.set_tag("optuna_study", "true")
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", MODEL_VERSION)
    
    logger.info("🚀 Comenzando optimización...")
    
    # Optuna optimization sin callback específico - MLflow tracking se maneja dentro de optuna_train
    study.optimize(optuna_train, n_trials=20)

    # Log resultados finales del estudio
    mlflow.log_metric("best_trial_number", study.best_trial.number)
    mlflow.log_metric("best_study_kappa", study.best_value)
    
    # Log mejores parámetros
    for param_name, param_value in study.best_params.items():
        mlflow.log_param(f"best_{param_name}", param_value)
    
    mlflow.set_tag("study_completed", "true")

logger.info("🎯 OPTIMIZACIÓN COMPLETADA")
logger.info(f"   ├── Mejor trial: #{study.best_trial.number}")
logger.info(f"   ├── Mejor kappa: {study.best_value:.6f}")
logger.info(f"   └── Mejores parámetros: {study.best_params}")

02:33:01 | INFO | 🔍 INICIANDO OPTIMIZACIÓN OPTUNA
02:33:01 | INFO |    ├── Proyecto: petfinder
02:33:01 | INFO |    ├── Modelo: 05 ResNet_1.0.0
02:33:01 | INFO |    ├── Base de datos: ../work/petfinder/db.sqlite3
02:33:01 | INFO |    └── Número de trials: 20
02:33:01 | INFO |    ├── Proyecto: petfinder
02:33:01 | INFO |    ├── Modelo: 05 ResNet_1.0.0
02:33:01 | INFO |    ├── Base de datos: ../work/petfinder/db.sqlite3
02:33:01 | INFO |    └── Número de trials: 20


[I 2025-08-17 02:33:01,073] Using an existing study with name '05 ResNet_1.0.0' instead of creating a new one.


02:33:01 | INFO | 🚀 Comenzando optimización...
02:33:01 | INFO | ============================================================
02:33:01 | INFO | ============================================================
02:33:01 | INFO | 🔬 INICIANDO OPTUNA TRIAL #0
02:33:01 | INFO | 🔬 INICIANDO OPTUNA TRIAL #0
02:33:01 | INFO |    ├── Hiperparámetros sugeridos:
02:33:01 | INFO |    ├──   Epochs: 2
02:33:01 | INFO |    ├──   Learning Rate: 0.000105
02:33:01 | INFO |    └──   Momentum: 0.2968
02:33:01 | INFO | ================================================================================
02:33:01 | INFO | 🚀 INICIANDO ENTRENAMIENTO - Trial 0
02:33:01 | INFO |    ├── Learning rate: 0.00010542416938670239
02:33:01 | INFO |    ├── Momentum: 0.2967909528278154
02:33:01 | INFO |    ├── Hiperparámetros sugeridos:
02:33:01 | INFO |    ├──   Epochs: 2
02:33:01 | INFO |    ├──   Learning Rate: 0.000105
02:33:01 | INFO |    └──   Momentum: 0.2968
02:33:01 | INFO | ===============================================

100%|██████████| 235/235 [11:03<00:00,  2.82s/it]

02:44:04 | INFO |       ├── 📈 TRAIN completado en 663.2s
02:44:04 | INFO |       ├── Loss: 1.339176
02:44:04 | INFO |       ├── Loss: 1.339176
02:44:04 | INFO |       ├── Accuracy: 38.00%
02:44:04 | INFO |       └── Kappa: N/A (entrenamiento)
02:44:04 | INFO |       ├── Accuracy: 38.00%
02:44:04 | INFO |       └── Kappa: N/A (entrenamiento)


...

Train Loss: 0.0684 Acc: 96.23% Kappa: nan


  0%|          | 0/59 [00:00<?, ?it/s]python(90168) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90168) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90169) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90169) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90173) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90173) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90174) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90178) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90174) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(90178) MallocStackLogging: can't turn off malloc stack logging because it

20:42:18 | INFO |       ├── 📈 VAL completado en 131.3s
20:42:18 | INFO |       ├── Loss: 2.939355
20:42:18 | INFO |       ├── Accuracy: 32.81%
20:42:18 | INFO |       └── Kappa: 0.258127
20:42:18 | INFO |       ├── Loss: 2.939355
20:42:18 | INFO |       ├── Accuracy: 32.81%
20:42:18 | INFO |       └── Kappa: 0.258127
Val Loss: 2.9394 Acc: 32.81% Kappa: 0.258
20:42:18 | INFO |    ⏱️  Época 2 completada en 948.4s
20:42:18 | INFO | 
20:42:18 | INFO | 🏁 ENTRENAMIENTO COMPLETADO
20:42:18 | INFO |    ├── ⏱️  Tiempo total: 31m 19s
20:42:18 | INFO |    ├── 🎯 Mejor Accuracy: 32.31%
20:42:18 | INFO |    ├── 🏆 Mejor Kappa: 0.262431
20:42:18 | INFO |    └── Trial 19 finalizado
Val Loss: 2.9394 Acc: 32.81% Kappa: 0.258
20:42:18 | INFO |    ⏱️  Época 2 completada en 948.4s
20:42:18 | INFO | 
20:42:18 | INFO | 🏁 ENTRENAMIENTO COMPLETADO
20:42:18 | INFO |    ├── ⏱️  Tiempo total: 31m 19s
20:42:18 | INFO |    ├── 🎯 Mejor Accuracy: 32.31%
20:42:18 | INFO |    ├── 🏆 Mejor Kappa: 0.262431
20:42:18 | INFO 


[I 2025-08-17 20:42:18,711] Trial 19 finished with value: 0.26243084129105987 and parameters: {'epochs': 2, 'lr': 0.00025505140797726835, 'momentum': 0.6930348097942659}. Best is trial 10 with value: 0.31846597159246404.
[I 2025-08-17 20:42:18,711] Trial 19 finished with value: 0.26243084129105987 and parameters: {'epochs': 2, 'lr': 0.00025505140797726835, 'momentum': 0.6930348097942659}. Best is trial 10 with value: 0.31846597159246404.


20:42:18 | INFO | 🎯 OPTIMIZACIÓN COMPLETADA
20:42:18 | INFO |    ├── Mejor trial: #10
20:42:18 | INFO |    ├── Mejor kappa: 0.318466
20:42:18 | INFO |    └── Mejores parámetros: {'epochs': 2, 'lr': 0.005261917062275208, 'momentum': 0.5338892393345371}
20:42:18 | INFO |    ├── Mejor trial: #10
20:42:18 | INFO |    ├── Mejor kappa: 0.318466
20:42:18 | INFO |    └── Mejores parámetros: {'epochs': 2, 'lr': 0.005261917062275208, 'momentum': 0.5338892393345371}
